#### Imports and setups
- We have to import required libraries and switches to the Gol schema and creates:
    - `gold_run_id`
    - a run date string
    - a run time stamp string
these are used for tracking and snapshot punblishing    

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable
from datetime import datetime
import uuid

In [0]:
spark.sql("""
          use catalog novacart_catalog
          """)


spark.sql("""
         CREATE SCHEMA IF NOT EXISTS novacart_catalog.gold
          """)

In [0]:
gold_run_id = str(uuid.uuid4())
run_ts_str = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
run_date_str = datetime.utcnow().strftime("%Y-%m-%d")

print(f"current gold run id: {gold_run_id}")
print("Current run timestamp Folder",run_ts_str)

#### Gold control table
This table stores the latest processing state:
- which silver data was processed last time
- how many gold rows were merged in the last run

In [0]:
spark.sql("""
          CREATE TABLE IF NOT EXISTS novacart_catalog.audit.gold_processing_control (
              layer STRING,
              table_name STRING,
              last_processed_silver_run_id STRING,
              last_processed_silver_ingested_at TIMESTAMP,
              rows_merged BIGINT,
              run_status STRING,
              gold_run_id STRING,
              updated_at TIMESTAMP
              )
              USING DELTA
          """)

#### Healper Functions
- `upsert_to_gold` merges data into gold tables
- `get_last_processed_silver_ts` reads the gold watermark from the control table
- `upsert_gold_control` Updates the gold control after every run

#### Read latest or changed silver rows only

this cell reads the full silver current state tables but filter out only rows that has changed only


In [0]:
from src.utils.gold_job_control import (
    upsert_to_gold, get_last_processed_silver_ts, upsert_gold_control
)

In [0]:
last_gold_ts = get_last_processed_silver_ts(spark,"orders_transformed")
print(last_gold_ts)

silver_orders_current = spark.read.table("novacart_catalog.silver.orders_transformed")
silver_products_current = spark.read.table("novacart_catalog.silver.products_transformed")
silver_payments_current = spark.read.table("novacart_catalog.silver.payments_transformed")

if last_gold_ts is None:
    changed_orders = silver_orders_current
    changed_products = silver_products_current
    changed_payments = silver_payments_current
else:
    changed_orders = silver_orders_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_products = silver_products_current.filter(F.col("updated_at") > F.lit(last_gold_ts))
    changed_payments = silver_payments_current.filter(F.col("updated_at") > F.lit(last_gold_ts))


changed_orders_count = changed_orders.count()
changed_products_count = changed_products.count()
changed_payments_count = changed_payments.count()

print(f"changed orders count: {changed_orders_count}")
print(f"changed products count: {changed_products_count}")
print(f"changed payments count: {changed_payments_count}")

    

#### Finding Impacted Order IDs

Gold is built at order grain, so if anything changed in orders, products, or payments, we identify with `order_id` values are impacted
only the impacted order_id are rebuilt in gold

In [0]:
impacted_from_orders = changed_orders.select("order_id").distinct()
impacted_from_payments = changed_payments.select("order_id").distinct()
impacted_from_products =  (
    changed_products.alias("p")
    .join(silver_orders_current.alias("o"),F.col("p.product_id") == F.col("o.product_id"),"inner")
    .select(F.col("o.order_id")).distinct()
)

impacted_order_ids = (
    impacted_from_orders
    .union(impacted_from_payments)
    .union(impacted_from_products)
    .distinct()
)

print("impacted order ids count",impacted_order_ids.count())
display(impacted_order_ids.orderBy("order_id"))


#### Gold delta for impacted orders
This cell joins the impacted orders with the current Silver Products and Payments tables, derives business columns, and builds the Gold delta that will be merged into the gold current state table

In [0]:
silver_payments_current.printSchema()

In [0]:
silver_products_current.printSchema()

In [0]:
impacted_order = (
    silver_orders_current.alias("o")
    .join(impacted_order_ids.alias("i"),"order_id","inner")
)

gold_delta = (
    impacted_order.alias("o")
    .join(silver_products_current.alias("p"),F.col("o.product_id") == F.col("p.product_id"),"inner")
    .join(silver_payments_current.alias("py"),F.col("o.order_id") == F.col("py.order_id"),"inner")
    .select(
        F.col("o.order_id"),
        F.col("o.customer_id"),
        F.col("p.product_id"),
        F.col("p.product_name"),
        F.col("p.category"),
        F.col("p.price").alias("product_price"),
        F.col("o.order_status"),
        F.col("o.order_amount"),
        F.col("py.payment_id"),
        F.col("py.payment_status"),
        F.col("py.paid_amount"),
        F.col("o.order_date"),
        F.col("o.order_month"),
        F.col("o.order_year"),
        F.greatest(
            F.col("o.updated_at").cast("timestamp"),
            F.col("p.updated_at").cast("timestamp"),
            F.col("py.processed_at").cast("timestamp"),
        ).alias("gold_updated_ts")
            )
    .dropDuplicates(["order_id"])
    .withColumn(
        "payment_completion_ratio",
        F.when(F.col("order_amount") > 0,F.col("paid_amount")/F.col("order_amount")
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "payment_state",
        F.when(F.col("order_amount") == 0,"invalid_order_amount")
        .when(F.col("payment_completion_ratio") == 0,"unpaid")
        .when(F.col("payment_completion_ratio") == 1,"paid")
        .when(F.col("payment_completion_ratio") < 1,"partially_paid")
        .when(F.col("payment_completion_ratio") > 1,"overpaid")
    )
    .withColumn(
        "gold_updated_date",
        F.to_date(F.col("gold_updated_ts"))
    )
    .withColumn("gold_run_id",F.lit(gold_run_id))
)

print("gold_delta_rows:",gold_delta.count())
display(gold_delta)



#### Merge Gold Current State Table
if the Gold delta contains rows, This cell merges them into `novacart_catalog.gold.orders_information`

if there is no impacted rows, nothing is merged

In [0]:
if gold_delta.count() > 0:
    upsert_to_gold(spark,gold_delta,"novacart_catalog.gold.orders_information","order_id")
else:
    print("no new rows to insert")


#### Maitaining Gold SCD Type 2 history
This cell updates the SCD type 2 history table
if the current Gold rows changes, the old version is closed(`is_current = false`) and new current version is inserted

In [0]:
if not spark.catalog.tableExists("novacart_catalog.gold.orders_information_scd2"):
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS novacart_catalog.gold.orders_information_scd2 
        USING DELTA
        AS
        SELECT *,
               cast(null as timestamp) as valid_from_ts,
               cast(null as timestamp) as valid_to_ts,
               true as is_current
        FROM novacart_catalog.gold.orders_information
        WHERE 1 = 0
            """)
    
if gold_delta.count() > 0:
    gold_delta.createOrReplaceTempView("gold_delta_view")
    spark.sql("""
              merge into novacart_catalog.gold.orders_information_scd2 t
              using gold_delta_view s
              on t.order_id = s.order_id and t.is_current = true
              when matched and (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price) 
                )
              then update set
                valid_to_ts = s.gold_updated_ts,
                is_current = false
            """)
    
    spark.sql("""
              insert into novacart_catalog.gold.orders_information_scd2
              select 
                s.*,
                s.gold_updated_ts as valid_from_ts,
                cast(null as timestamp) as valid_to_ts,
                true as is_current
              from gold_delta_view s
              left join novacart_catalog.gold.orders_information_scd2 t
              on s.order_id = t.order_id and t.is_current = true
              where t.order_id is null or (
                  not(t.order_status <=> s.order_status) or
                  not(t.order_amount <=> s.order_amount) or
                  not(t.paid_amount <=> s.paid_amount) or
                  not(t.payment_id <=> s.payment_id) or
                  not(t.category <=> s.category) or
                  not(t.product_name <=> s.product_name) or
                  not(t.product_price <=> s.product_price) 
                )
              """)
 




#### Update category level Gold Aggregatons

This cell calculates the category level business metrics only for categories impacted in the current run, then merges them into category performance Gold table

In [0]:
if gold_delta.count() > 0:
    impacted_categories = (
        gold_delta
        .select("category")
        .filter(F.col("category").isNotNull())
        .distinct()
        )
    category_pref_delta = (
        spark.read.table("novacart_catalog.gold.orders_information")
        .join(impacted_categories,"category","inner")
        .groupBy("category")
        .agg(
            F.countDistinct("order_id").alias("total_orders"),
            F.sum(
                F.when(F.col("order_amount") > 0,F.col("order_amount"))
                .otherwise(F.lit(0.0)))
            .alias("Gross_Merchandise_Value"),
            F.sum(
                F.when(F.col("paid_amount") > 0, F.col("paid_amount"))
                .otherwise(F.lit(0.0)))
            .alias("Total_Paid_Amount"),
            F.avg("payment_completion_ratio").alias("Average_Payment_Completion_Ratio"),
            (
                F.sum(F.when(F.col("payment_status") == 'FAILED',1).otherwise(0)) / F.count("*")
            ).alias("Failed_Payments_Rate")
            )
        )
    
    upsert_to_gold(spark,category_pref_delta,"novacart_catalog.gold.category_preference","category")
            

#### Publish Gold snapshots to Volumes

this cell writes 2 kinds of gold outputs to a databricks volume:

-   latest snapshot -> overwritten every succesful run
-   timestampedhistoricalsnapshot -> a new folder for each succesful run

this is useful for audit and rollback

In [0]:
spark.sql("""
          CREATE VOLUME IF NOT EXISTS novacart_catalog.gold.gold_snapshots_vol
          """)

In [0]:
latest_orders_path = (
    "/Volumes/novacart_catalog/gold/gold_snapshots_vol/gold_latest/orders_information"
)

latest_category_path = (
    "/Volumes/novacart_catalog/gold/gold_snapshots_vol/gold_latest/category_preference"
)

historical_orders_path = f"/Volumes/novacart_catalog/gold/gold_snapshots_vol/gold_snapshots/orders_information/run_date={run_date_str}/run_ts={run_ts_str}"

historical_category_path = f"/Volumes/novacart_catalog/gold/gold_snapshots_vol/gold_snapshots/category_preference/run_date={run_date_str}/run_ts={run_ts_str}"

spark.read.table("novacart_catalog.gold.orders_information").write.mode("overwrite").format("parquet").save(latest_orders_path)

spark.read.table("novacart_catalog.gold.category_preference").write.mode("overwrite").format("parquet").save
(latest_category_path)

spark.read.table("novacart_catalog.gold.orders_information").write.mode("overwrite").format("parquet").save(historical_orders_path)

spark.read.table("novacart_catalog.gold.category_preference").write.mode("overwrite").format("parquet").save
(historical_category_path)

print("latest orders path",latest_orders_path)
print("latest category path",latest_category_path)
print("historical orders path",historical_orders_path)
print("historical category path",historical_category_path)



#### Update Gold Control table

This final cell updates the Gold Control table using the latest Silver processing metadata


In [0]:
latest_siver_ts = silver_orders_current.agg(F.max("bronze_ingested_at").alias("max")).collect()[0]["max"]

latest_silver_run_id = (
    silver_orders_current
    .filter(F.col("bronze_ingested_at") == latest_siver_ts)
    .agg(F.max("silver_run_id").alias("max"))
    .collect()[0]["max"]
) if latest_siver_ts is not None else None


upsert_gold_control(
    spark,
    "novacart_catalog.gold.orders_information",
    latest_silver_run_id,
    latest_siver_ts,
    gold_delta.count(),
    gold_run_id   
)

display(
    spark.read.table("novacart_catalog.audit.gold_processing_control")
    )